# Previsão de Resultado de Partidas de League of Legends

## Projeto Final — Módulo 43 | Case 03 — Riot Games

---

### 🎯 Objetivo do Projeto
Desenvolver um **modelo de Machine Learning** capaz de prever o **time vencedor** de uma partida de League of Legends com base em dados coletados durante os primeiros minutos de jogo (estado pós-início de partida).
 
### 📋 Escopo Técnico
| Item | Detalhe |
|------|---------|
| **Problema** | Classificação binária — Time Azul vence (1) ou perde (0) |
| **Dados** | ~9.879 partidas ranqueadas com 40 variáveis |
| **Modelos** | Regressão Logística, Naive Bayes, Árvore de Decisão, Random Forest, Gradient Boosting |
| **Métrica principal** | AUC-ROC (solicitado pelo stakeholder) |
| **Bônus** | Cross-validation, Ensemble, GridSearchCV, App Streamlit |
 
### ✍️ Índice
1. Setup e Importações
2. Carregamento e Entendimento dos Dados
3. Análise Exploratória de Dados (EDA)
4. Preparação dos Dados e Feature Engineering
5. Modelagem
6. Avaliação Comparativa
7. Ajuste de Hiperparâmetros (Tuning)
8. Insights, Conclusões e Recomendações

# 1. Setup e Importações

In [5]:
# Bibliotecas de manipulação de dados
import pandas as pd
import numpy as np

# Visualização de dados
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

# Scikit-learn: pré-processamento
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler

# Scikit-learn: modelos
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Scikit-learn: métricas
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report, ConfusionMatrixDisplay

# Persistência de modelos
import joblib
import os

In [6]:
# Configurações globais de visualização
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'grid.linewidth':   0.7,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
    'axes.labelsize':   12,
    'xtick.labelsize':  10,
    'ytick.labelsize':  10,
    'legend.fontsize':  11,
})
sns.set_palette('deep')

In [7]:
# Constantes do projeto
BLUE        = '#1f77b4'   # Time Azul — vitória
RED         = '#d62728'   # Time Vermelho — derrota
GREEN       = '#2ca02c'   # Métricas positivas
AMBER       = '#ff7f0e'   # Destaques / atenção
PURPLE      = '#9467bd'   # Ensemble
RANDOM_STATE = 42

In [8]:
# Pastas de saída
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# 2. Carregamento e Entendimento dos Dados
 
### 📖 Dicionário de Variáveis (40 colunas)
 
O dataset registra o estado da partida em um ponto pós-início de jogo (~10 min). Toda variável com prefixo **`blue`** tem uma equivalente em **`red`**, exceto a target. Total: 1 identificador + 1 target + 19 features × 2 times = 40 colunas.
 
| Variável (prefixo `blue`/`red`) | Tipo | Descrição |
|----------------------------------|------|-----------|
| `gameId` | int | Identificador único da partida *(coluna única, sem prefixo)* |
| `blueWins` | int (0/1) | **TARGET** — 1 se o Time Azul venceu *(coluna única, sem equivalente red)* |
| `WardsPlaced` | int | Wards colocadas (visão no mapa) |
| `WardsDestroyed` | int | Wards inimigas destruídas |
| `FirstBlood` | int (0/1) | Se o time obteve o primeiro abate da partida |
| `Kills` | int | Total de campeões abatidos |
| `Deaths` | int | Total de mortes do time (= Kills do time adversário) |
| `Assists` | int | Total de assistências em abates |
| `EliteMonsters` | int | Monstros elite capturados (Dragões + Arautos somados) |
| `Dragons` | int | Dragões capturados |
| `Heralds` | int | Arautos (Heralds) capturados |
| `TowersDestroyed` | int | Torres inimigas destruídas |
| `TotalGold` | int | Ouro total acumulado pelo time |
| `AvgLevel` | float | Nível médio dos campeões do time |
| `TotalExperience` | int | Experiência total acumulada pelo time |
| `TotalMinionsKilled` | int | Total de minions (creeps de lane) abatidos — *farm* |
| `TotalJungleMinionsKilled` | int | Total de monstros de selva abatidos — *farm de jungle* |
| `GoldDiff` | int | Diferença de ouro em relação ao time adversário (blue − red) |
| `ExperienceDiff` | int | Diferença de experiência em relação ao adversário (blue − red) |
| `CSPerMin` | float | CS (minions + jungle) por minuto — eficiência de farm |
| `GoldPerMin` | float | Ouro ganho por minuto |
 
> ⚠️ Note que `redGoldDiff` = −`blueGoldDiff` e `redExperienceDiff` = −`blueExperienceDiff` (espelhos exatos), e `blueDeaths` = `redKills` enquanto `redDeaths` = `blueKills`. Essas redundâncias serão tratadas na seção de Preparação dos Dados.

In [16]:
# carregar o dataset
DATASET_PATH = "data/raw/Base_M43_Pratique_LOL_RANKED_WIN.csv"
df = pd.read_csv(DATASET_PATH)
df

,gameId,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,...,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
0,4519157822,0,28,2,1,9,6,11,0,0,...,0,16567,6.8,17047,197,55,-643,8,19.7,1656.7
1,4523371949,0,12,1,0,5,5,5,0,0,...,1,17620,6.8,17438,240,52,2908,1173,24.0,1762.0
2,4521474530,0,15,0,0,7,11,4,1,1,...,0,17285,6.8,17254,203,28,1172,1033,20.3,1728.5
3,4524384067,0,43,1,0,4,5,5,1,0,...,0,16478,7.0,17961,235,47,1321,7,23.5,1647.8
4,4436033771,0,75,4,0,6,6,6,0,0,...,0,17404,7.0,18313,225,67,1004,-230,22.5,1740.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9874,4527873286,1,17,2,1,7,4,5,1,1,...,0,15246,6.8,16498,229,34,-2519,-2469,22.9,1524.6
9875,4527797466,1,54,0,0,6,4,8,1,1,...,0,15456,7.0,18367,206,56,-782,-888,20.6,1545.6
9876,4527713716,0,23,1,0,6,7,5,0,0,...,0,18319,7.4,19909,261,60,2416,1877,26.1,1831.9
9877,4527628313,0,14,4,1,2,3,3,1,1,...,0,15298,7.2,18314,247,40,839,1085,24.7,1529.8


In [30]:
# Tipos de dados
print(df.dtypes.value_counts().to_string())
df.dtypes

int64      34
float64     6


gameId                            int64
blueWins                          int64
blueWardsPlaced                   int64
blueWardsDestroyed                int64
blueFirstBlood                    int64
blueKills                         int64
blueDeaths                        int64
blueAssists                       int64
blueEliteMonsters                 int64
blueDragons                       int64
blueHeralds                       int64
blueTowersDestroyed               int64
blueTotalGold                     int64
blueAvgLevel                    float64
blueTotalExperience               int64
blueTotalMinionsKilled            int64
blueTotalJungleMinionsKilled      int64
blueGoldDiff                      int64
blueExperienceDiff                int64
blueCSPerMin                    float64
blueGoldPerMin                  float64
redWardsPlaced                    int64
redWardsDestroyed                 int64
redFirstBlood                     int64
redKills                          int64


In [25]:
# Valores nulos
df.isnull().sum()

gameId                          0
blueWins                        0
blueWardsPlaced                 0
blueWardsDestroyed              0
blueFirstBlood                  0
blueKills                       0
blueDeaths                      0
blueAssists                     0
blueEliteMonsters               0
blueDragons                     0
blueHeralds                     0
blueTowersDestroyed             0
blueTotalGold                   0
blueAvgLevel                    0
blueTotalExperience             0
blueTotalMinionsKilled          0
blueTotalJungleMinionsKilled    0
blueGoldDiff                    0
blueExperienceDiff              0
blueCSPerMin                    0
blueGoldPerMin                  0
redWardsPlaced                  0
redWardsDestroyed               0
redFirstBlood                   0
redKills                        0
redDeaths                       0
redAssists                      0
redEliteMonsters                0
redDragons                      0
redHeralds    

Nenhum valor nulo encontrado! Dataset completamente limpo.

In [23]:
# Estatísticas descritivas (amostra)
df[["blueKills", "blueTotalGold", "blueGoldDiff", "blueExperienceDiff",
    "blueDragons", "blueHeralds"]].describe().round(2)

,blueKills,blueTotalGold,blueGoldDiff,blueExperienceDiff,blueDragons,blueHeralds
count,9879.00,9879.00,9879.00,9879.00,9879.00,9879.00
mean,6.18,16503.46,14.41,-33.62,0.36,0.19
std,3.01,1535.45,2453.35,1920.37,0.48,0.39
min,0.00,10730.00,-10830.00,-9333.00,0.00,0.00
25%,4.00,15415.50,-1585.50,-1290.50,0.00,0.00
50%,6.00,16398.00,14.00,-28.00,0.00,0.00
75%,8.00,17459.00,1596.00,1212.00,1.00,0.00
max,22.00,23701.00,11467.00,8348.00,1.00,1.00


In [32]:
# Análise da variável target (blueWins)
var_target = df["blueWins"].value_counts().sort_index()

print("Distribuição da variável target (blueWins):\n")
print(f'  Derrota do Time Azul (0): {var_target[0]:,} ({var_target[0]/len(df):.1%})')
print(f'  Vitória do Time Azul  (1): {var_target[1]:,} ({var_target[1]/len(df):.1%})')

Distribuição da variável target (blueWins):

  Derrota do Time Azul (0): 4,949 (50.1%)
  Vitória do Time Azul  (1): 4,930 (49.9%)


Dataset Balanceado (≈ 50/50)

Sem necessidade de SMOTE, under-sampling ou class weighting
Accuracy é uma métrica válida aqui (além do AUC-ROC)

# 3. Análise Exploratória de Dados (EDA)